<a href="https://colab.research.google.com/github/NavanidhiDJ/6thSem-ML-Lab/blob/main/1BM23CS204_Lab_10_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

df = pd.read_csv("heart.csv")

X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

categorical_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
numerical_cols = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first'), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

def evaluate(use_pca=False):
    results = {}

    for name, model in models.items():
        if use_pca:
            pipeline = Pipeline([
                ('preprocessing', preprocessor),
                ('pca', PCA(n_components=5)),
                ('model', model)
            ])
        else:
            pipeline = Pipeline([
                ('preprocessing', preprocessor),
                ('model', model)
            ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        results[name] = acc

        print(f"{name} | PCA={use_pca} → Accuracy: {acc:.4f}")

    return results

print("\nWITHOUT PCA")
results_no_pca = evaluate(use_pca=False)

print("\nWITH PCA")
results_with_pca = evaluate(use_pca=True)

best_no_pca = max(results_no_pca, key=results_no_pca.get)
best_with_pca = max(results_with_pca, key=results_with_pca.get)

print("\nBest WITHOUT PCA:", best_no_pca, results_no_pca[best_no_pca])
print("Best WITH PCA:", best_with_pca, results_with_pca[best_with_pca])


WITHOUT PCA
Logistic Regression | PCA=False → Accuracy: 0.8533
SVM | PCA=False → Accuracy: 0.8587
Random Forest | PCA=False → Accuracy: 0.8750

WITH PCA
Logistic Regression | PCA=True → Accuracy: 0.8098
SVM | PCA=True → Accuracy: 0.7989
Random Forest | PCA=True → Accuracy: 0.7717

Best WITHOUT PCA: Random Forest 0.875
Best WITH PCA: Logistic Regression 0.8097826086956522
